In [1]:
from __future__ import annotations

import os
import math
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr


# VAE Model

In [2]:
class VAE(nn.Module):
    """A simple multi‑layer perceptron VAE for grayscale images.

    The encoder maps a flattened image x to a latent mean and log‑variance.
    The decoder reconstructs the image from a latent vector z.  This
    architecture deliberately avoids convolutional layers to keep the code
    lightweight; feel free to substitute a convolutional VAE for better
    performance on real data【886131745861763†L37-L52】.
    """

    def __init__(self, input_dim: int, latent_dim: int = 10, hidden_dims: List[int] | None = None) -> None:
        super().__init__()
        if hidden_dims is None:
            # Default hidden layer sizes
            hidden_dims = [512, 256]
        self.latent_dim = latent_dim

        # Build encoder
        encoder_layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            encoder_layers.append(nn.Linear(prev_dim, h))
            encoder_layers.append(nn.ReLU())
            prev_dim = h
        # Final layers for mean and log variance
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_mu = nn.Linear(prev_dim, latent_dim)
        self.fc_logvar = nn.Linear(prev_dim, latent_dim)

        # Build decoder (symmetrical to encoder)
        decoder_layers = []
        prev_dim = latent_dim
        for h in reversed(hidden_dims):
            decoder_layers.append(nn.Linear(prev_dim, h))
            decoder_layers.append(nn.ReLU())
            prev_dim = h
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        decoder_layers.append(nn.Sigmoid())  # outputs in [0,1]
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode input images into latent mean and log variance."""
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample latent variable z using the reparameterization trick."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent variables back into image space."""
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Run a forward pass through the VAE.

        Returns
        -------
        reconstruction : torch.Tensor
            Reconstructed images in [0,1].
        mu : torch.Tensor
            Latent means.
        logvar : torch.Tensor
            Latent log variances.
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar



# Training and Latent Extraction

In [3]:
@dataclass
class TrainingConfig:
    """Hyperparameters for VAE training."""
    epochs: int = 10
    batch_size: int = 128
    learning_rate: float = 1e-3
    device: str = "cpu"


def loss_function(recon_x: torch.Tensor, x: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """Compute the VAE loss = reconstruction loss + KL divergence.

    Uses binary cross‑entropy (BCE) for reconstruction and KL divergence of
    latent distribution q(z|x) from the prior p(z) = N(0,I)【886131745861763†L37-L52】.
    """
    bce = F.binary_cross_entropy(recon_x, x, reduction="sum")
    # KL divergence between N(mu, sigma^2) and N(0,1)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld


def train_vae(model: VAE, dataloader: DataLoader, config: TrainingConfig) -> None:
    """Train the VAE on the provided dataset.

    Parameters
    ----------
    model : VAE
        The variational autoencoder to train.
    dataloader : DataLoader
        DataLoader yielding batches of flattened images in [0,1].
    config : TrainingConfig
        Hyperparameters such as number of epochs and learning rate.
    """
    model.to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    model.train()
    for epoch in range(config.epochs):
        total_loss = 0.0
        for batch, _ in dataloader:
            batch = batch.to(config.device)
            optimizer.zero_grad()
            recon_x, mu, logvar = model(batch)
            loss = loss_function(recon_x, batch, mu, logvar)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{config.epochs}, Loss: {avg_loss:.4f}")


def extract_latent_codes(model: VAE, dataloader: DataLoader, device: str = "cpu") -> Tuple[np.ndarray, np.ndarray]:
    """Extract latent means from a trained VAE for all data points.

    Parameters
    ----------
    model : VAE
        Trained VAE model.
    dataloader : DataLoader
        DataLoader returning (image, label) pairs. Images must be flattened
        already.  Labels are used for computing supervised directions (e.g.
        difference‑of‑means).
    device : str, optional
        Device on which to perform computation (default: cpu).

    Returns
    -------
    z : np.ndarray, shape (N, latent_dim)
        Extracted latent means for each sample.
    labels : np.ndarray, shape (N,)
        Corresponding labels.
    """
    model.eval()
    model.to(device)
    zs: List[np.ndarray] = []
    ys: List[int] = []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            mu, _ = model.encode(x)
            zs.append(mu.cpu().numpy())
            ys.append(y.cpu().numpy())
    z_arr = np.concatenate(zs, axis=0)
    y_arr = np.concatenate(ys, axis=0)
    return z_arr, y_arr



# Direction Discovery

In [4]:
def normalize_direction(v: np.ndarray) -> np.ndarray:
    """Normalize a direction vector to unit L2 norm.  Returns zero vector unchanged."""
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm


def direction_difference_of_means(z: np.ndarray, labels: np.ndarray, positive_label: int, negative_label: Optional[int] = None) -> np.ndarray:
    """Compute the difference‑of‑means direction for a binary attribute【886131745861763†L65-L72】.

    Parameters
    ----------
    z : np.ndarray, shape (N, D)
        Latent codes.
    labels : np.ndarray, shape (N,)
        Attribute labels; either the class labels or binary labels.
    positive_label : int
        Label indicating the positive class (attribute present).
    negative_label : int, optional
        Label for the negative class; if None, uses all labels != positive_label.

    Returns
    -------
    direction : np.ndarray, shape (D,)
        Normalized difference of means vector v = mean(z_pos) − mean(z_neg).
    """
    pos_idx = labels == positive_label
    neg_idx = labels != positive_label if negative_label is None else labels == negative_label
    if not np.any(pos_idx) or not np.any(neg_idx):
        raise ValueError("Insufficient samples for computing difference‑of‑means.")
    mean_pos = z[pos_idx].mean(axis=0)
    mean_neg = z[neg_idx].mean(axis=0)
    v = mean_pos - mean_neg
    return normalize_direction(v)


def direction_lda(z: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Compute a discriminative direction using Linear Discriminant Analysis【886131745861763†L69-L72】.

    This function fits an LDA model and returns the first discriminant vector,
    normalized to unit length.  It expects at least two classes in `labels`.
    """
    lda = LinearDiscriminantAnalysis(n_components=1)
    lda.fit(z, labels)
    # Coefficients shape (n_features, n_components)
    v = lda.scalings_[:, 0]
    return normalize_direction(v.real)


def direction_svm(z: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Compute a discriminative direction using a linear Support Vector Machine【886131745861763†L69-L72】.

    Fits a linear SVM (hinge loss) and returns the weight vector.
    """
    # Standardize features before SVM for numerical stability
    scaler = StandardScaler()
    z_std = scaler.fit_transform(z)
    svm = LinearSVC(C=1.0, loss="hinge", max_iter=10000)
    svm.fit(z_std, labels)
    # Weight vector in standardized space needs to be mapped back to original scale
    v_std = svm.coef_.flatten()
    # Rescale: w_original = w_std / std_dev
    v = v_std / scaler.scale_
    return normalize_direction(v)


def direction_pca(z: np.ndarray, n_components: int = 1) -> np.ndarray:
    """Compute principal directions of latent space using PCA【886131745861763†L75-L76】.

    Parameters
    ----------
    z : np.ndarray, shape (N, D)
        Latent codes.
    n_components : int, optional
        Number of principal components to compute (default: 1).

    Returns
    -------
    directions : np.ndarray, shape (n_components, D)
        The top `n_components` principal directions, normalized to unit length.
    """
    pca = PCA(n_components=n_components)
    pca.fit(z)
    components = pca.components_
    return np.array([normalize_direction(v) for v in components])


def direction_gradient(model: nn.Module, classifier: nn.Module, dataloader: DataLoader, device: str = "cpu") -> np.ndarray:
    """Compute a gradient‑based direction for an attribute score【886131745861763†L73-L74】.

    A simple approach is to train a small probe (classifier) on the VAE latent
    codes to predict an attribute, then use the gradient of the probe's
    output with respect to the latent input.  Here we assume `classifier` is
    already trained and maps latent vectors to a scalar score via a Sigmoid
    activation.  The returned direction is the average gradient over a batch of
    samples.

    Parameters
    ----------
    model : nn.Module
        Trained VAE model whose encoder will produce latent codes.
    classifier : nn.Module
        Probe network mapping latent vectors to attribute scores (Sigmoid
        probabilities).
    dataloader : DataLoader
        DataLoader yielding (image, label) pairs.  Only images are used to
        compute latent codes.
    device : str, optional
        Device on which to perform computation (default: cpu).

    Returns
    -------
    direction : np.ndarray, shape (latent_dim,)
        Normalized average gradient direction.
    """
    model.eval()
    classifier.eval()
    model.to(device)
    classifier.to(device)
    grads: List[torch.Tensor] = []
    for x, _ in dataloader:
        x = x.to(device)
        x.requires_grad = False
        mu, _ = model.encode(x)
        mu = mu.detach().requires_grad_(True)
        scores = classifier(mu).squeeze()
        # We want gradient of mean score
        mean_score = scores.mean()
        mean_score.backward()
        grad = mu.grad.detach().cpu()
        grads.append(grad)
        # Only compute gradient on first batch for efficiency
        break
    grad_mat = torch.cat(grads, dim=0)
    v = grad_mat.mean(dim=0).numpy()
    return normalize_direction(v)


# Local Steering via Clustering

In [5]:

def cluster_latent_codes(z: np.ndarray, n_clusters: int) -> np.ndarray:
    """Cluster latent codes using k‑means and return cluster assignments【886131745861763†L78-L83】."""
    kmeans = KMeans(n_clusters=n_clusters, n_init="auto", random_state=0)
    assignments = kmeans.fit_predict(z)
    return assignments


def compute_local_directions(z: np.ndarray, labels: np.ndarray, assignments: np.ndarray, method: str = "difference_of_means") -> Dict[int, np.ndarray]:
    """Compute a direction per cluster using the specified method.

    Supported methods include ``difference_of_means``, ``lda``, ``svm`` and
    ``pca``.  For PCA the same top component is used for all clusters.

    Parameters
    ----------
    z : np.ndarray, shape (N, D)
        Latent codes.
    labels : np.ndarray, shape (N,)
        Labels or attribute values.
    assignments : np.ndarray, shape (N,)
        Cluster assignment for each sample.
    method : str, optional
        Method used to compute directions within each cluster (default:
        ``"difference_of_means"``).

    Returns
    -------
    directions : dict
        Mapping from cluster index to normalized direction vector.
    """
    directions: Dict[int, np.ndarray] = {}
    unique_clusters = np.unique(assignments)
    for cluster_id in unique_clusters:
        idx = assignments == cluster_id
        z_cluster = z[idx]
        labels_cluster = labels[idx]
        if method == "difference_of_means":
            # Use majority label as positive and the rest as negative
            # Identify the most frequent label
            unique_labels, counts = np.unique(labels_cluster, return_counts=True)
            pos_label = unique_labels[np.argmax(counts)]
            directions[cluster_id] = direction_difference_of_means(z_cluster, labels_cluster, pos_label)
        elif method == "lda":
            directions[cluster_id] = direction_lda(z_cluster, labels_cluster)
        elif method == "svm":
            directions[cluster_id] = direction_svm(z_cluster, labels_cluster)
        elif method == "pca":
            directions[cluster_id] = direction_pca(z_cluster, n_components=1)[0]
        else:
            raise ValueError(f"Unknown method: {method}")
    return directions


# Steering and Constraint‑Based Step Selection

In [6]:
def steer_latent(z: np.ndarray, direction: np.ndarray, alpha: float) -> np.ndarray:
    """Apply a steering step to a latent vector: z' = z + alpha * v.【886131745861763†L83-L99】"""
    return z + alpha * direction


def constraint_based_alpha(
    z: np.ndarray,
    direction: np.ndarray,
    decoder: Callable[[torch.Tensor], torch.Tensor],
    attribute_fn: Callable[[torch.Tensor], torch.Tensor],
    content_penalty_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    alphas: List[float],
    threshold: float,
    device: str = "cpu",
) -> float:
    """Select an alpha that improves attribute score while limiting content change【886131745861763†L86-L99】.

    For each candidate alpha, we decode z' = z + alpha * v and compute
    attribute score s(z') and content penalty d(z,z').  The first alpha that
    yields a positive increase in attribute score and has penalty <= threshold
    is returned.  If none satisfy the constraints, returns zero (no change).

    Parameters
    ----------
    z : np.ndarray, shape (D,)
        Original latent vector.
    direction : np.ndarray, shape (D,)
        Direction vector v (should be unit length).
    decoder : callable
        Function mapping a latent tensor to a reconstructed image tensor.
    attribute_fn : callable
        Function mapping an image tensor to a scalar attribute score; higher
        values correspond to stronger presence of the attribute.
    content_penalty_fn : callable
        Function computing a penalty between two images; e.g., mean squared
        error or feature distance.
    alphas : list of float
        Grid of candidate step sizes to consider; should include zero and
        increasing positive values.
    threshold : float
        Maximum allowed penalty for a step to be considered safe.
    device : str, optional
        Device on which to perform computations (default: cpu).

    Returns
    -------
    alpha_star : float
        Selected step size.  Returns 0.0 if no safe improvement exists.
    """
    # Ensure direction is normalized
    v = normalize_direction(direction)
    # Encode original latent as torch tensor
    z_t = torch.tensor(z, dtype=torch.float32, device=device)
    # Decode original image
    with torch.no_grad():
        x0 = decoder(z_t.unsqueeze(0)).detach().cpu()
        s0 = attribute_fn(x0).item()
    for alpha in alphas:
        if alpha == 0:
            continue
        z_new = z_t + alpha * torch.tensor(v, dtype=torch.float32, device=device)
        with torch.no_grad():
            x_new = decoder(z_new.unsqueeze(0)).detach().cpu()
            s_new = attribute_fn(x_new).item()
            penalty = content_penalty_fn(x0, x_new).item()
        # Accept step if attribute improves and penalty is within threshold
        if s_new > s0 and penalty <= threshold:
            return alpha
    # No safe improvement found
    return 0.0



# Evaluation Metrics

In [7]:
def evaluate_monotonicity(
    scores: np.ndarray,
    alphas: np.ndarray,
) -> float:
    """Compute Spearman correlation between attribute scores and step sizes【886131745861763†L110-L118】."""
    if len(scores) < 2:
        return 0.0
    rho, _ = spearmanr(alphas, scores)
    return rho


def evaluate_content_preservation(
    original_labels: np.ndarray,
    steered_labels: np.ndarray,
) -> float:
    """Compute the fraction of samples whose class label remains unchanged【886131745861763†L116-L118】."""
    return np.mean(original_labels == steered_labels)


def evaluate_smoothness(
    distances: np.ndarray,
) -> float:
    """Return mean squared distance as a proxy for smoothness【886131745861763†L119-L122】."""
    return distances.mean()



# Example Usage

In [9]:
def example_dataset(n_samples: int = 1000, input_dim: int = 784, n_classes: int = 2) -> Tuple[Dataset, Dataset]:
    """Create a toy dataset of random binary images and labels.

    This function returns training and validation datasets consisting of
    random binary vectors to exercise the VAE pipeline without requiring
    external data downloads.  Each sample is a vector of length ``input_dim``
    with values in {0,1}.  Labels are random integers in ``[0, n_classes)``.
    """
    class RandomBinaryDataset(Dataset):
        def __init__(self, n: int) -> None:
            self.x = torch.bernoulli(0.5 * torch.ones(n, input_dim))
            self.y = torch.randint(0, n_classes, (n,))
        def __len__(self) -> int:
            return len(self.x)
        def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
            return self.x[idx], int(self.y[idx])
    total = n_samples
    # Use an 80/20 train/val split
    train_ds = RandomBinaryDataset(int(0.8 * total))
    val_ds = RandomBinaryDataset(total - int(0.8 * total))
    return train_ds, val_ds


def main() -> None:
    """Run a demonstration of the latent steering pipeline on a toy dataset.

    When using real datasets such as MNIST or Fashion‑MNIST, replace
    ``example_dataset`` with appropriate dataset loaders and adjust
    ``input_dim`` accordingly (e.g., 28*28=784 for MNIST).  The demonstration
    trains a VAE for a few epochs, extracts latent codes, computes a
    difference‑of‑means direction on a binary attribute, and then steers
    several samples along that direction.  Metrics are printed to illustrate
    the evaluation procedure.  Real experiments should use the full training
    set and evaluate on hold‑out data【886131745861763†L110-L131】.
    """
    # Generate a toy dataset
    train_ds, val_ds = example_dataset(n_samples=500, input_dim=784, n_classes=2)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Define and train VAE
    latent_dim = 6
    model = VAE(input_dim=784, latent_dim=latent_dim)
    config = TrainingConfig(epochs=5, batch_size=64, learning_rate=1e-3)
    print("Training VAE on toy data...")
    train_vae(model, train_loader, config)

    # Extract latent codes and labels from validation set
    print("Extracting latent codes...")
    z, labels = extract_latent_codes(model, val_loader)

    # Compute a global difference‑of‑means direction for label 1 vs others
    print("Computing global direction (difference of means)...")
    direction = direction_difference_of_means(z, labels, positive_label=1)

    # Cluster latent codes and compute local directions
    print("Clustering latent codes and computing local directions...")
    assignments = cluster_latent_codes(z, n_clusters=3)
    local_directions = compute_local_directions(z, labels, assignments, method="difference_of_means")

    # Define a simple attribute score: probability of being class 1 using a logistic regression on pixels
    # In real experiments, use a pretrained classifier on images【886131745861763†L116-L118】.
    class PixelClassifier(nn.Module):
        def __init__(self, input_dim: int):
            super().__init__()
            self.fc = nn.Linear(input_dim, 1)
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = x.view(x.size(0), -1)
            return torch.sigmoid(self.fc(x))
    # Train the pixel classifier quickly on toy data
    classifier = PixelClassifier(784)
    clf_optimizer = torch.optim.SGD(classifier.parameters(), lr=1e-1)
    classifier.train()
    for epoch in range(3):
        for batch_x, batch_y in train_loader:
            clf_optimizer.zero_grad()
            preds = classifier(batch_x).squeeze()
            loss = F.binary_cross_entropy(preds, batch_y.float())
            loss.backward()
            clf_optimizer.step()
    classifier.eval()

    # Attribute function using pixel classifier on decoded images
    def attribute_fn(x: torch.Tensor) -> torch.Tensor:
        return classifier(x).squeeze()

    # Content penalty: mean squared error between images
    def content_penalty(x0: torch.Tensor, x1: torch.Tensor) -> torch.Tensor:
        return F.mse_loss(x0, x1)

    # Evaluate steering on a few validation samples
    print("Evaluating steering with constraint‑based alpha selection...")
    alphas_grid = [0.0] + [0.2 * i for i in range(1, 6)]  # candidate step sizes
    threshold = 0.05
    monotonicities: List[float] = []
    label_consistencies: List[float] = []
    smoothnesses: List[float] = []
    # Use a simple nearest‑centroid classifier to infer labels from images
    # For toy data we treat pixel sum as proxy for class
    def infer_label(x: torch.Tensor) -> int:
        return int((x.sum() > (x.numel() / 2)).item())
    for i in range(min(10, len(z))):
        z_i = z[i]
        label_i = labels[i]
        cluster_id = assignments[i]
        v = local_directions[cluster_id]
        # Evaluate attribute scores for each alpha
        alpha_scores: List[float] = []
        alpha_list: List[float] = []
        img0 = model.decode(torch.tensor(z_i, dtype=torch.float32)).detach().view(1, -1)
        original_label = infer_label(img0)
        for alpha in alphas_grid[1:]:
            z_new = steer_latent(z_i, v, alpha)
            with torch.no_grad():
                img_new = model.decode(torch.tensor(z_new, dtype=torch.float32)).detach().view(1, -1)
                score = attribute_fn(img_new).item()
                alpha_scores.append(score)
                alpha_list.append(alpha)
        # Compute monotonicity
        mono = evaluate_monotonicity(np.array(alpha_scores), np.array(alpha_list))
        monotonicities.append(mono)
        # Choose alpha via constraint‑based selection
        alpha_star = constraint_based_alpha(
            z_i,
            v,
            decoder=lambda z_t: model.decode(z_t),
            attribute_fn=attribute_fn,
            content_penalty_fn=content_penalty,
            alphas=alphas_grid,
            threshold=threshold,
            device="cpu",
        )
        # Apply selected alpha
        if alpha_star > 0:
            z_new = steer_latent(z_i, v, alpha_star)
            img_new = model.decode(torch.tensor(z_new, dtype=torch.float32)).detach().view(1, -1)
            new_label = infer_label(img_new)
            penalty = content_penalty(img0, img_new).item()
        else:
            new_label = original_label
            penalty = 0.0
        # Evaluate label consistency and smoothness
        label_consistencies.append(int(original_label == new_label))
        smoothnesses.append(penalty)
    print(f"Average monotonicity (Spearman rho): {np.mean(monotonicities):.3f}")
    print(f"Class consistency: {np.mean(label_consistencies):.3f}")
    print(f"Average smoothness penalty: {np.mean(smoothnesses):.5f}")


if __name__ == "__main__":
    main()

Training VAE on toy data...
Epoch 1/5, Loss: 544.3346
Epoch 2/5, Loss: 543.2045
Epoch 3/5, Loss: 542.9369
Epoch 4/5, Loss: 542.8941
Epoch 5/5, Loss: 542.8012
Extracting latent codes...
Computing global direction (difference of means)...
Clustering latent codes and computing local directions...
Evaluating steering with constraint‑based alpha selection...
Average monotonicity (Spearman rho): 0.550
Class consistency: 1.000
Average smoothness penalty: 0.00001


 # On mnist

In [10]:
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

DATASET = "mnist"  # "fashion_mnist"
BATCH_TRAIN = 128
BATCH_VAL = 256
VAL_RATIO = 0.2

tfm = transforms.ToTensor()

ds_cls = datasets.MNIST if DATASET == "mnist" else datasets.FashionMNIST
full = ds_cls(root="./data", train=True, download=True, transform=tfm)

n_val = int(len(full) * VAL_RATIO)
n_train = len(full) - n_val
train_raw, val_raw = random_split(full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

def flatten_loader(ds, batch_size, shuffle):
    def collate(batch):
        x = torch.stack([b[0] for b in batch]).view(len(batch), -1)  # (B,784)
        y = torch.tensor([b[1] for b in batch], dtype=torch.long)
        return x, y
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

train_loader = flatten_loader(train_raw, BATCH_TRAIN, shuffle=True)
val_loader   = flatten_loader(val_raw,   BATCH_VAL,   shuffle=False)

input_dim = 28 * 28
device = "cuda" if torch.cuda.is_available() else "cpu"


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 346kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.21MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.10MB/s]


In [11]:
latent_dim = 10
model = VAE(input_dim=input_dim, latent_dim=latent_dim).to(device)

config = TrainingConfig(
    epochs=10,
    batch_size=BATCH_TRAIN,
    learning_rate=1e-3,
    device=device,
)

train_vae(model, train_loader, config)


Epoch 1/10, Loss: 180.5115
Epoch 2/10, Loss: 128.2438
Epoch 3/10, Loss: 118.0162
Epoch 4/10, Loss: 113.6557
Epoch 5/10, Loss: 111.0632
Epoch 6/10, Loss: 109.3066
Epoch 7/10, Loss: 108.0031
Epoch 8/10, Loss: 107.0124
Epoch 9/10, Loss: 106.1506
Epoch 10/10, Loss: 105.4613


In [12]:
z, y = extract_latent_codes(model, val_loader, device=device)  # z: (N,latent_dim), y: (N,)

target_class = 1
a = (y == target_class).astype(np.int64)  # бинарный атрибут 0/1


In [14]:
v_global = direction_difference_of_means(z, a, positive_label=1, negative_label=0)

K = 5
assign = cluster_latent_codes(z, n_clusters=K)

try:
    v_local = compute_local_directions(z, a, assign, method="difference_of_means")
except ValueError:
    # Если в каком-то кластере нет обеих групп (a=0 и a=1), используем global direction везде
    v_local = {int(k): v_global for k in np.unique(assign)}


In [15]:
import torch.nn.functional as F

@torch.no_grad()
def decode_np(z_np):
    z_t = torch.tensor(z_np, dtype=torch.float32, device=device).unsqueeze(0)
    x = model.decode(z_t).detach().cpu().view(1, -1)  # (1,784)
    return x

def attribute_fn(x_flat):
    # супер-простой прокси score: средняя интенсивность
    return x_flat.mean(dim=1)  # (B,)

def content_penalty(x0, x1):
    return F.mse_loss(x0, x1)

alphas = [0.0] + [0.2 * i for i in range(1, 11)]
thr = 0.02

monos, smooth = [], []
n_eval = min(50, len(z))

for i in range(n_eval):
    z_i = z[i]
    k = int(assign[i])
    v = v_local[k]

    x0 = decode_np(z_i)

    # monotonicity along grid (excluding 0)
    scores, alpha_list = [], []
    for a_step in alphas[1:]:
        x1 = decode_np(steer_latent(z_i, v, a_step))
        scores.append(float(attribute_fn(x1).item()))
        alpha_list.append(a_step)
    monos.append(evaluate_monotonicity(np.array(scores), np.array(alpha_list)))

    # constraint-based alpha*
    a_star = constraint_based_alpha(
        z_i, v,
        decoder=lambda zt: model.decode(zt.to(device)),
        attribute_fn=lambda x: attribute_fn(x.detach().cpu()),
        content_penalty_fn=lambda x0_, x1_: content_penalty(x0_, x1_),
        alphas=alphas,
        threshold=thr,
        device=device,
    )

    x_best = decode_np(steer_latent(z_i, v, a_star)) if a_star > 0 else x0
    smooth.append(float(content_penalty(x0, x_best).item()))

print(f"Monotonicity mean (Spearman rho): {np.mean(monos):.3f}")
print(f"Smoothness mean (pixel MSE):      {np.mean(smooth):.5f}")


Monotonicity mean (Spearman rho): -0.878
Smoothness mean (pixel MSE):      0.00001


# TBD: fix the parameters and training